# 0 – BreastGNN: Main Pipeline

Entrenamiento completo del modelo GNN para clasificación de subtipos moleculares de cáncer de mama.

In [1]:
import os, sys
sys.path.insert(0, "..")  # ajustar si breastgnn está en otra ruta


from breastgnn.config import CFG
from breastgnn.utils import set_seed, cleanup_memory, get_device
from breastgnn.data import (
    load_expression_and_metadata, prepare_genes, encode_labels,
    cohort_split, scale_features, apply_connected_only, make_dataloaders,
    build_regulator_features,
)
from breastgnn.graph import build_backbone
from breastgnn.graph_cache import get_or_build_backbone, get_or_build_Xh
from breastgnn.losses import FocalLoss, compute_metrics_full, compute_class_weights_balanced
from breastgnn.training import predict_proba_xh_mode, train_graph_learning, finetune_pruned
from breastgnn.pruning import export_pruned_graph, evaluate_keep_ratios
from breastgnn.stability import edge_set_from_edge_index, pairwise_jaccard_stats
from breastgnn.ablation import (
    AblationConfig, build_model_from_cfg, make_prior_for_cfg,
    run_single_seed, run_ablation, register_runtime,
)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

set_seed(CFG.SEED)
DEVICE = CFG.DEVICE
print("DEVICE:", DEVICE)


DEVICE: cuda


## 1) Configuración

Modifica aquí los hiperparámetros si es necesario.

In [2]:
# ── Override defaults if needed ──
# CFG.DATA_DIR = Path("../Listados_tumores/output_combat/").resolve()
# CFG.LR = 1e-3
# CFG.EPOCHS1 = 90
print("Config OK")

Config OK


## 2) Cargar datos + genes

In [3]:
X_df, y_str, cohort = load_expression_and_metadata(
    CFG.EXPR_CSV, CFG.META_CSV,
    sample_col=CFG.SAMPLE_COL, label_col=CFG.LABEL_COL, cohort_col=CFG.COHORT_COL,
)
X_df_kegg, genes_kegg = prepare_genes(X_df)
y, classes, label_map = encode_labels(y_str)
n_classes = len(classes)
print("Classes:", classes)

X_df: (5812, 11907) | y: (5812,) | cohort: (5812,)
Targets:
 LumA      2514
LumB      1202
TNBC      1026
HER2       602
Normal     468
Name: count, dtype: int64
Genes totales (nodos): 11907
y: (5812,), n_classes: 5, classes: ['HER2', 'LumA', 'LumB', 'Normal', 'TNBC']
Classes: ['HER2', 'LumA', 'LumB', 'Normal', 'TNBC']


## 3) Grafo backbone (HuRI + OmniPath) con caché

Usa `get_or_build_backbone` (graph_cache) que envuelve `build_backbone` (graph.py).
Si existe un caché validado se carga directamente; si no, se reconstruye
desde HuRI.psi + OmniPath TSV, se deduplicican aristas y se guarda.

La validación comprueba:
- **REQUIRED_CONNECTED_GENES** (FAM118A, TMEM30A, TMEM30B)
- **EXPECTED_CONNECTED_GENES** (referencia `genes_usados.csv` si existe)


In [4]:
# ── Opción A: con doble capa de caché (graph_cache → build_backbone) ──
# get_or_build_backbone valida + cachea en pipeline_cache/
# y build_backbone a su vez cachea en backbone_cache/.
#
# Opción B (directa): build_backbone ya tiene su propio caché.
# Descomenta la línea B y comenta la A si prefieres un solo nivel.

# --- A) Doble caché (recomendada, replica el flujo del notebook Grafo) ---
edge_index, edge_weight, edge_type = get_or_build_backbone(
    genes_kegg,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    force_rebuild=CFG.FORCE_REBUILD_HURI_CACHE,
    use_omnipath=CFG.USE_OMNIPATH,
    use_huri=CFG.USE_HURI,
)

# --- B) Caché directo (solo backbone_cache/) ---
# edge_index, edge_weight, edge_type = build_backbone(
#     genes_kegg,
#     use_omnipath=CFG.USE_OMNIPATH,
#     use_huri=CFG.USE_HURI,
#     force_rebuild=CFG.FORCE_REBUILD_HURI_CACHE,
# )

print(f"Backbone: {edge_index.shape[1]} aristas, {len(genes_kegg)} genes")


[graph_cache] Generando backbone (build_backbone)…
[Backbone cache] loaded: backbone_global__n11907__sig5f9c12d26c9c90d5__op1__huri1__score1__min0p0.npz
[Backbone HuRI+OmniPath] connected genes (deg>0): 7213 (60.6%)
[graph_cache] Backbone guardado en: /home/alejandro/Escritorio/Trabajo/Laura_Senovilla/Grafo_TNBC/breastgnn_package/Listados_tumores/output_combat/pipeline_cache/backbone__362de41006ca05c1__e71ae7463734.npz
[graph_cache]   → 51445 aristas
Backbone: 51445 aristas, 11907 genes


## 4) Split + escalado + connected-only

In [5]:
train_idx, val_idx, test_idx = cohort_split(
    cohort, y, train_cohort_frac=0.80, val_size=CFG.VAL_SIZE, seed=CFG.SEED,
)
Xs_gene = scale_features(X_df_kegg, train_idx, mode=CFG.SCALE_MODE, use_quantile=CFG.USE_QUANTILE)

if CFG.CONNECTED_ONLY:
    Xs_gene, edge_index, edge_weight, edge_type, genes_kegg = apply_connected_only(
        Xs_gene, edge_index, edge_weight, edge_type, genes_kegg,
    )

Total cohorts: 12
Split sizes: train=3805, val=952, test=1055
Cohorts TRAIN: ['GSE1456', 'GSE15852', 'GSE162228', 'GSE19615', 'GSE20711', 'GSE21653', 'GSE25066', 'GSE65194', 'GSE96058']
Cohorts TEST: ['GSE32646', 'GSE58812', 'TCGA']
[CONNECTED_ONLY] 11907 -> 7213 genes, 51445 -> 51445 edges


## 5) Features de reguladores (Xs_graph) con caché

Usa `get_or_build_Xh` (graph_cache) o directamente `build_regulator_features`.
Las features dependen de Xs_gene (escalado), genes conectados y edge_index,
así que un cambio en split/semilla/escalado invalida automáticamente el caché.


In [6]:
# --- A) Con caché (recomendada, replica el flujo del notebook Grafo) ---
Xs_graph, graph_feat_names = get_or_build_Xh(
    Xs_gene, genes_kegg, edge_index,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    force_rebuild=False,
    stats=CFG.REG_STATS,
    min_targets=CFG.REG_MIN_GENES,
    max_regulators=CFG.REG_MAX_REGULATORS,
)

# --- B) Sin caché (directo) ---
# Xs_graph, graph_feat_names = build_regulator_features(
#     Xs_gene, genes_kegg, edge_index,
#     add_features=CFG.ADD_REGULATOR_FEATURES,
#     stats=CFG.REG_STATS,
#     min_genes=CFG.REG_MIN_GENES,
#     max_regulators=CFG.REG_MAX_REGULATORS,
# )

print(f"Xs_graph: {Xs_graph.shape} | features: {len(graph_feat_names)}")


[graph_cache] Generando Xs_graph (X_h) por primera vez…
[data] Regulators used for X_graph: 2537 (min_targets=5)
[data] X_graph shape: (5812, 7611)
[graph_cache] Xs_graph guardado en: /home/alejandro/Escritorio/Trabajo/Laura_Senovilla/Grafo_TNBC/breastgnn_package/Listados_tumores/output_combat/pipeline_cache/Xh__0575558be8d8a450__c102b7893b38__2060915c8ab9.npz
[graph_cache]   → shape: (5812, 7611)
Xs_graph: (5812, 7611) | features: 7611


## 6) DataLoaders

In [7]:
dl_tr, dl_va, dl_te = make_dataloaders(
    Xs_gene, Xs_graph, y, train_idx, val_idx, test_idx,
    batch_size=CFG.BATCH_SIZE,
)

Batches train/val/test: 191/48/53


## 7) Tensores de grafo

In [8]:
edge_index_t = torch.as_tensor(edge_index, dtype=torch.long, device=DEVICE)
edge_weight_t = torch.as_tensor(edge_weight, dtype=torch.float32, device=DEVICE)
edge_type_t = torch.as_tensor(edge_type, dtype=torch.long, device=DEVICE)

# Adjacency sparse (para sparse_mm_batch)
N = Xs_gene.shape[1]
adj = torch.sparse_coo_tensor(
    edge_index_t, edge_weight_t, size=(N, N),
).coalesce().to(DEVICE)
print("adj:", adj.shape)

adj: torch.Size([7213, 7213])


## 8) Ablación + entrenamiento

In [9]:
register_runtime(
    Xs_gene=Xs_gene, genes_kegg=genes_kegg,
    X_h=Xs_graph, y=y, n_classes=n_classes, label_map=label_map,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
    edge_index_t=edge_index_t, edge_weight_t=edge_weight_t,
    edge_type_t=edge_type_t,
    reg_groups_t=reg_groups_t if 'reg_groups_t' in dir() else None,
)

In [10]:
# Configuración de ablación (solo FULL por defecto)
CFG_FULL = AblationConfig(
    name="FULL",
    edge_type_gating=CFG.EDGE_TYPE_GATING,
    sample_cond_gating=CFG.SAMPLE_COND_GATING,
    sample_cond_mode=CFG.SAMPLE_COND_MODE,
    use_signed_prior=True,
    signed_channels_mode=CFG.SIGNED_CHANNELS_MODE,
    connectivity_penalty=CFG.ADD_CONNECTIVITY_PENALTY,
    do_pretrain=CFG.DO_PRETRAIN,
    do_stability=CFG.DO_STABILITY_SELECTION,
)

ABLA_CONFIGS = [CFG_FULL]
ABLA_SEEDS = [1234, 1, 42, 369]

# Ejecutar ablación
df_abla = run_ablation(ABLA_CONFIGS, ABLA_SEEDS, save_root=CFG.ARTIFACTS_ROOT)
display(df_abla)




=== FULL ===
  - seed 1234
[TRAIN hard] {'keep_ratio': 1.0, 'thr_logit': 0.0, 'thr_gate': 0.5, 'thr': 0.5, 'kept': 51445, 'total': 51445}


KeyboardInterrupt: 

In [ ]:
# ── Ablaciones individuales (descomenta las que quieras)
ABLA_CONFIGS = [
    CFG_FULL,
    AblationConfig("no_conn",        True,  True,  "per_type", True,  "type_only",     False, CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("typed_only",     True,  False, "per_type", True,  "type_only",     True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("cond_only",      False, True,  "global",   True,  "type_only",     True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("no_type_no_cond",False, False, "global",   True,  "type_only",     True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("no_signed",      True,  True,  "per_type", False, "type_only",     True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    AblationConfig("dual_backbone",  True,  True,  "per_type", True,  "dual_backbone", True,  CFG.DO_PRETRAIN, CFG.DO_STABILITY_SELECTION),
    # AblationConfig("no_pretrain",    True,  True,  "per_type", True,  "type_only",     True,  False,           CFG.DO_STABILITY_SELECTION),
    # AblationConfig("no_stability",   True,  True,  "per_type", True,  "type_only",     True,  CFG.DO_PRETRAIN, False),
]

ABLA_SEEDS = [1234, 1, 42, 369]

df_abla = run_ablation(ABLA_CONFIGS, ABLA_SEEDS, save_root=CFG.ARTIFACTS_ROOT)
display(df_abla)

In [ ]:
# Resumen
summary = (
    df_abla.groupby("cfg")[["macro_f1","auc_ovr_macro","acc","n_edges_final","n_nodes_compact"]]
    .mean()
    .sort_values("macro_f1", ascending=False)
)
display(summary)

## 9) Post-hoc: cargar modelo ganador y evaluar

In [11]:
from breastgnn.ablation import load_bundle

bundle_dir = f"{CFG.ARTIFACTS_ROOT}/FULL/seed_1234"
bundle = load_bundle(bundle_dir)
print("Bundle cargado:", list(bundle.keys()) if isinstance(bundle, dict) else "OK")

Bundle cargado: ['meta', 'graph', 'model']
